In [1]:
import os
import re
import string

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

from sklearn.metrics import(
    accuracy_score,
    classification_report,
    confusion_matrix
)

from collections import Counter


In [2]:
path = 'Datasets' 
csv_path = os.path.join(path,"IMDB Dataset.csv")

df = pd.read_csv(csv_path)

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
print(df.shape)

(50000, 2)


In [4]:
import re

email_pattern = r"[\w.-]+@[\w.-]+\.\w+"
url_pattern = r"https?://\S+\.\S"

URL = re.compile(url_pattern)
EMAIL = re.compile(email_pattern)

In [5]:
# Removal of Stop words

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk

nltk.download("stopwords")
nltk.download("punkt")

stop_words = stopwords.words("english")


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/mukundanramesh/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/mukundanramesh/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [6]:
def text_clean(cell):
  if not isinstance(cell,str):
    return " "

  cell = EMAIL.sub(" ",cell)
  cell = URL.sub(" ",cell)
  cell = cell.lower()
  cell = re.sub(r"\d+"," ",cell)
  cell = re.sub(r"[^\w\s]","",cell)

  cell = [word for word in cell.split() if word not in stop_words]
  cell = " ".join(cell)

  return cell

df["clean_review"] = df["review"].apply(text_clean)

df.head()

,review,sentiment,clean_review
0,One of the other reviewers has mentioned that ...,positive,one reviewers mentioned watching oz episode yo...
1,A wonderful little production. <br /><br />The...,positive,wonderful little production br br filming tech...
2,I thought this was a wonderful way to spend ti...,positive,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,negative,basically theres family little boy jake thinks...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter matteis love time money visually stunni...


In [7]:
df["label"] = df["sentiment"].map(
    {
        "positive":1,
        "negative":0
    }
)

df.head()

,review,sentiment,clean_review,label
0,One of the other reviewers has mentioned that ...,positive,one reviewers mentioned watching oz episode yo...,1
1,A wonderful little production. <br /><br />The...,positive,wonderful little production br br filming tech...,1
2,I thought this was a wonderful way to spend ti...,positive,thought wonderful way spend time hot summer we...,1
3,Basically there's a family where a little boy ...,negative,basically theres family little boy jake thinks...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter matteis love time money visually stunni...,1


In [8]:
train_texts, test_texts, train_labels,test_labels = train_test_split(
    df["clean_review"].values,
    df["label"].values,
    test_size = 0.2,
    random_state = 42
)

print("Train texts shape:",train_texts.shape)
print("Test texts shape:",test_texts.shape)

Train texts shape: (40000,)
Test texts shape: (10000,)


In [ ]:
# Build Vocab

MAX_VOCAB_SIZE = 20000

word_counter = Counter()

for text in train_texts:
  words = text.split()
  word_counter.update(words)

most_common_words = word_counter.most_common(
    MAX_VOCAB_SIZE - 2
)

word_to_index = {
    "<PAD>":0,
    "<UNK>":1
}

for word, _ in most_common_words:
  word_to_index[word] = len(word_to_index)


print("Vocabulary size:",len(word_to_index))